# Práctica de Laboratorio: AES y Modos de Operación sobre Imágenes BMP
**Objetivo:** Comprender de forma visual y práctica cómo funcionan los modos de operación ECB, CBC y CTR al aplicarse sobre una imagen BMP

## 1. Arquitectura del Archivo BMP y Configuración
Para poder visualizar los efectos del cifrado, no podemos cifrar el archivo completo. Los archivos BMP tienen una estructura específica donde los primeros 54 bytes corresponden a la cabecera (header) que contiene metadatos vitales (dimensiones, paleta de colores)Si ciframos esta sección, el sistema operativo no reconocerá el archivo como una imagen

Por lo tanto, la arquitectura de nuestra solución dividirá el archivo: preservaremos los primeros 54 bytes intactos y aplicaremos las transformaciones criptográficas únicamente al cuerpo (los píxeles)

In [ ]:
import json
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes

def pad(data, block_size=16):
   
    pad_len = block_size - (len(data) % block_size)
    return data + bytes([pad_len]) * pad_len

def read_bmp(file_path):
    
    with open(file_path, "rb") as f:
        bmp = f.read()
        header = bmp[:54]
        body = bmp[54:]
    return header, body

def write_bmp(file_path, header, body):
    
    with open(file_path, "wb") as f:
        f.write(header + body)

## 2. Gestión del Estado Criptográfico (Material)
En sistemas de producción, las llaves y vectores de inicialización nunca se "queman" en el código. Aquí implementamos un gestor de estado que genera entropía real usando `get_random_bytes` y persiste estos parámetros en un archivo JSON. 

* **Key:** 16 bytes para AES-128.
* **IV:** 16 bytes, requerido para el modo CBC.
* **Nonce:** 8 bytes, requerido para el modo CTR.

In [ ]:
def generate_crypto_material():
    material = {
        "key": get_random_bytes(16).hex(),
        "iv": get_random_bytes(16).hex(),
        "nonce": get_random_bytes(8).hex()
    }
    return material

def save_crypto_material(material, output_file):
    with open(output_file, "w") as f:
        json.dump(material, f, indent=4)

def load_crypto_material(input_file):
    with open(input_file, "r") as f:
        material = json.load(f)
    return material

# Ejecución: Generar y guardar
archivo_claves = "mis_claves_MATRICULA.json"
material_cripto = generate_crypto_material()
save_crypto_material(material_cripto, archivo_claves)
print(f"[+] Material criptográfico guardado en: {archivo_claves}")

[+] Material criptográfico guardado en: mis_claves_MATRICULA.json
